## Estruturação da camada **SILVER** -  Limpeza e padronização

Principais alterações:
- Linhas duplicadas (exatamente iguais) removidas
- Linhas sem identificador (id em branco/nulo) removidas


**Tabela:** orders

Formatação de valores:
- Valores de dinheiro (order_amount, order_delivery_fee, order_delivery_cost) foram ajustados para DECIMAL(10,2)
- Tempos de duração (order_metric_*) foram ajustados para DECIMAL(10,2)
- Colunas de data/hora (order_moment_*) que estavam como texto foram ajustadas para TIMESTAMP

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.orders AS
SELECT DISTINCT
    order_id,
    store_id,
    channel_id,
    order_status,
    CAST(order_amount AS DECIMAL(10,2))        AS order_amount,
    CAST(order_delivery_fee AS DECIMAL(10,2))  AS order_delivery_fee,
    CAST(order_delivery_cost AS DECIMAL(10,2)) AS order_delivery_cost,
    order_created_day,
    order_created_month,
    order_created_year,
    try_to_timestamp(order_moment_created, 'M/d/yyyy h:mm:ss a')       AS order_moment_created,
    try_to_timestamp(order_moment_accepted, 'M/d/yyyy h:mm:ss a')      AS order_moment_accepted,
    try_to_timestamp(order_moment_ready, 'M/d/yyyy h:mm:ss a')         AS order_moment_ready,
    try_to_timestamp(order_moment_collected, 'M/d/yyyy h:mm:ss a')     AS order_moment_collected,
    try_to_timestamp(order_moment_in_expedition, 'M/d/yyyy h:mm:ss a') AS order_moment_in_expedition,
    try_to_timestamp(order_moment_delivering, 'M/d/yyyy h:mm:ss a')    AS order_moment_delivering,
    try_to_timestamp(order_moment_delivered, 'M/d/yyyy h:mm:ss a')     AS order_moment_delivered,
    try_to_timestamp(order_moment_finished, 'M/d/yyyy h:mm:ss a')      AS order_moment_finished,
    CAST(order_metric_collected_time AS DECIMAL(10,2))       AS order_metric_collected_time,
    CAST(order_metric_paused_time AS DECIMAL(10,2))          AS order_metric_paused_time,
    CAST(order_metric_production_time AS DECIMAL(10,2))      AS order_metric_production_time,
    CAST(order_metric_walking_time AS DECIMAL(10,2))         AS order_metric_walking_time,
    CAST(order_metric_expediton_speed_time AS DECIMAL(10,2)) AS order_metric_expediton_speed_time,
    CAST(order_metric_transit_time AS DECIMAL(10,2))         AS order_metric_transit_time,
    CAST(order_metric_cycle_time AS DECIMAL(10,2))           AS order_metric_cycle_time
FROM delivery_center.bronze.orders
WHERE order_id IS NOT NULL;

**Tabela:** deliveries

Formatação de valores:
- Distância da entrega (delivery_distance_meters) foi ajustada para BIGINT

Deduplicação:
- Encontrados 9.531 pedidos com mais de um registro de entrega (mesmo delivery_order_id) — provavelmente reatribuições de motorista
- Mantida 1 linha por delivery_order_id, priorizando status DELIVERED sobre CANCELLED (um pedido reentregue com sucesso conta como entregue), com driver_id como desempate quando o status é igual

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.deliveries AS
SELECT delivery_order_id, driver_id, delivery_status, delivery_distance_meters
FROM (
    SELECT
        delivery_order_id,
        driver_id,
        delivery_status,
        CAST(delivery_distance_meters AS BIGINT) AS delivery_distance_meters,
        ROW_NUMBER() OVER (
            PARTITION BY delivery_order_id
            ORDER BY CASE WHEN delivery_status = 'DELIVERED' THEN 0
                          WHEN delivery_status = 'CANCELLED' THEN 2
                          ELSE 1 END,
                     driver_id
        ) AS rn
    FROM delivery_center.bronze.deliveries
    WHERE delivery_order_id IS NOT NULL
)
WHERE rn = 1;

**Tabela:** stores

Formatação de valores:
- Valor do plano da loja (store_plan_price) foi ajustado para DECIMAL(10,2)

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.stores AS
SELECT DISTINCT
    store_id,
    store_name,
    store_segment,
    CAST(store_plan_price AS DECIMAL(10,2)) AS store_plan_price,
    store_latitude,
    store_longitude,
    hub_id
FROM delivery_center.bronze.stores
WHERE store_id IS NOT NULL;

**Tabela:** hubs

Formatação de valores:
- Nenhum ajuste adicional de tipo foi necessário

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.hubs AS
SELECT DISTINCT
    hub_id, hub_name, hub_city, hub_state, hub_latitude, hub_longitude
FROM delivery_center.bronze.hubs
WHERE hub_id IS NOT NULL;


**Tabela:** drivers

Formatação de valores:
- Nenhum ajuste adicional de tipo foi necessário

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.drivers AS
SELECT DISTINCT driver_id, driver_modal, driver_type
FROM delivery_center.bronze.drivers
WHERE driver_id IS NOT NULL;

**Tabela:** channels

Formatação de valores:
- Nenhum ajuste adicional de tipo foi necessário

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.channels AS
SELECT DISTINCT channel_id, channel_name, channel_type
FROM delivery_center.bronze.channels
WHERE channel_id IS NOT NULL;

**Tabela:** payments

Formatação de valores:
- Valor pago (payment_amount) foi ajustado para DECIMAL(10,2)
- Taxa do pagamento (payment_fee) foi ajustada para DECIMAL(10,2)

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.silver.payments AS
SELECT DISTINCT
    payment_id,
    payment_order_id,
    CAST(payment_amount AS DECIMAL(10,2)) AS payment_amount,
    CAST(payment_fee AS DECIMAL(10,2))    AS payment_fee,
    payment_method,
    payment_status
FROM delivery_center.bronze.payments
WHERE payment_id IS NOT NULL;

## Estruturação da camada **GOLD** - Modelagem em Esquema Estrela

Principais alterações:
- Tabelas relacionadas unidas (joins), formando as dimensões e fatos do modelo
- Chave substituta (date_id) criada para a dimensão de data
- Colunas renomeadas para seguir o padrão do modelo (ex.: payment_order_id → order_id)

**Tabela:** dim_stores

Estruturação:
- stores e hubs unidas (LEFT JOIN por hub_id), trazendo os dados do hub para dentro da dimensão de loja

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.dim_stores AS
SELECT
    s.store_id,
    s.store_name,
    s.store_segment,
    s.store_plan_price,
    s.store_latitude,
    s.store_longitude,
    h.hub_id,
    h.hub_name,
    h.hub_city,
    h.hub_state,
    h.hub_latitude,
    h.hub_longitude
FROM delivery_center.silver.stores s
LEFT JOIN delivery_center.silver.hubs h ON s.hub_id = h.hub_id;

**Tabela:** dim_drivers

Estruturação:
- Cópia direta da tabela silver.drivers, sem alterações

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.dim_drivers AS
SELECT driver_id, driver_modal, driver_type
FROM delivery_center.silver.drivers;

**Tabela:** dim_channels

Estruturação:
- Cópia direta da tabela silver.channels, sem alterações

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.dim_channels AS
SELECT channel_id, channel_name, channel_type
FROM delivery_center.silver.channels;

**Tabela:** dim_date

Estruturação:
- Combinações únicas de dia/mês/ano de orders selecionadas (DISTINCT)
- Chave substituta (date_id) criada para cada combinação, usando ROW_NUMBER()

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.dim_date AS
SELECT
    ROW_NUMBER() OVER (ORDER BY order_created_year, order_created_month, order_created_day) AS date_id,
    order_created_day   AS day,
    order_created_month AS month,
    order_created_year  AS year
FROM (
    SELECT DISTINCT order_created_day, order_created_month, order_created_year
    FROM delivery_center.silver.orders
);

**Tabela:** fact_orders

Estruturação:
- orders e deliveries unidas (LEFT JOIN por order_id = delivery_order_id), trazendo driver_id, delivery_status e delivery_distance_meters
- date_id resolvido casando dia/mês/ano do pedido com a dim_date
- Colunas order_created_day/month/year removidas daqui — substituídas pelo date_id

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.fact_orders AS
SELECT
    o.order_id,
    o.store_id,
    d.driver_id,
    o.channel_id,
    dt.date_id,
    o.order_status,
    o.order_amount,
    o.order_delivery_fee,
    o.order_delivery_cost,
    o.order_moment_created,
    o.order_moment_accepted,
    o.order_moment_ready,
    o.order_moment_collected,
    o.order_moment_in_expedition,
    o.order_moment_delivering,
    o.order_moment_delivered,
    o.order_moment_finished,
    o.order_metric_collected_time,
    o.order_metric_paused_time,
    o.order_metric_production_time,
    o.order_metric_walking_time,
    o.order_metric_expediton_speed_time,
    o.order_metric_transit_time,
    o.order_metric_cycle_time,
    d.delivery_status,
    d.delivery_distance_meters
FROM delivery_center.silver.orders o
LEFT JOIN delivery_center.silver.deliveries d
    ON o.order_id = d.delivery_order_id
LEFT JOIN delivery_center.gold.dim_date dt
    ON o.order_created_day = dt.day
   AND o.order_created_month = dt.month
   AND o.order_created_year = dt.year;

**Tabela:** fact_payments

Estruturação:
- Cópia da tabela silver.payments, com payment_order_id renomeado para order_id

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_center.gold.fact_payments AS
SELECT
    payment_id,
    payment_order_id AS order_id,
    payment_amount,
    payment_fee,
    payment_method,
    payment_status
FROM delivery_center.silver.payments;

✅ **Confirmação da criação das tabelas**

In [0]:
%sql
SELECT 'silver.orders' AS tabela, COUNT(*) AS linhas FROM delivery_center.silver.orders
UNION ALL SELECT 'silver.deliveries', COUNT(*) FROM delivery_center.silver.deliveries
UNION ALL SELECT 'silver.stores', COUNT(*) FROM delivery_center.silver.stores
UNION ALL SELECT 'silver.hubs', COUNT(*) FROM delivery_center.silver.hubs
UNION ALL SELECT 'silver.drivers', COUNT(*) FROM delivery_center.silver.drivers
UNION ALL SELECT 'silver.channels', COUNT(*) FROM delivery_center.silver.channels
UNION ALL SELECT 'silver.payments', COUNT(*) FROM delivery_center.silver.payments
UNION ALL SELECT 'gold.dim_stores', COUNT(*) FROM delivery_center.gold.dim_stores
UNION ALL SELECT 'gold.dim_drivers', COUNT(*) FROM delivery_center.gold.dim_drivers
UNION ALL SELECT 'gold.dim_channels', COUNT(*) FROM delivery_center.gold.dim_channels
UNION ALL SELECT 'gold.dim_date', COUNT(*) FROM delivery_center.gold.dim_date
UNION ALL SELECT 'gold.fact_orders', COUNT(*) FROM delivery_center.gold.fact_orders
UNION ALL SELECT 'gold.fact_payments', COUNT(*) FROM delivery_center.gold.fact_payments;